# IVECO Statistics - VODR Subset

Notebook temporaneo per generare solo alcuni sheet VODR, senza join Mission Test e senza dipendere da `mission` o `mileage_range`.

In [ ]:
import os

import pyspark.sql.functions as F

from run_local_sample import (
    build_spark,
    copy_excel_to_dbfs,
    ensure_excel_writer_available,
    export_excel_outputs,
)
from engine_cleaning import (
    add_legacy_preparation_features,
    clean_spark_column_names,
    keep_latest_record_per_vin,
)
from engine_loader import get_table_path
from engine_stats import pyspark_variabili_x
from engine_utils import log_step, report_dim, report_vin
from vodr_config import VODR_EXCLUDED_VINS, VODR_PERCENTAGE_GROUPS
from vodr_pipeline import (
    DEFAULT_VODR_INPUT_MODE,
    DEFAULT_VODR_SAMPLE_FORMAT,
    DEFAULT_VODR_SAMPLE_PATH,
    build_vodr_sheet_output,
    load_variables_names,
    parse_config_text,
    read_vodr_input_data,
)

## 0. Widget e configurazione

In [ ]:
DEFAULT_SUBSET_CONFIG = {56}
DEFAULT_SUBSET_OUTPUT_DIR = "/tmp/iveco_vodr_subset_output"
DEFAULT_SUBSET_DBFS_OUTPUT_DIR = "dbfs:/FileStore/iveco_vodr_subset_output"

default_config_text = ",".join(map(str, sorted(DEFAULT_SUBSET_CONFIG)))

try:
    dbutils
    try:
        dbutils.widgets.get("config")
    except Exception:
        dbutils.widgets.text("config", default_config_text, "Config VODR")
    try:
        dbutils.widgets.get("input_mode")
    except Exception:
        dbutils.widgets.dropdown("input_mode", DEFAULT_VODR_INPUT_MODE, ["fat_table", "sample"], "Input mode")
    try:
        dbutils.widgets.get("keep_latest_per_vin")
    except Exception:
        dbutils.widgets.dropdown("keep_latest_per_vin", "Yes", ["Yes", "No"], "Latest per VIN")
    try:
        dbutils.widgets.get("export_excel")
    except Exception:
        dbutils.widgets.dropdown("export_excel", "Yes", ["Yes", "No"], "Export Excel")
    try:
        dbutils.widgets.get("output_dir")
    except Exception:
        dbutils.widgets.text("output_dir", DEFAULT_SUBSET_OUTPUT_DIR, "Output dir")
    try:
        dbutils.widgets.get("sample_path")
    except Exception:
        dbutils.widgets.text("sample_path", DEFAULT_VODR_SAMPLE_PATH, "Sample path")
    try:
        dbutils.widgets.get("input_format")
    except Exception:
        dbutils.widgets.dropdown("input_format", DEFAULT_VODR_SAMPLE_FORMAT, ["parquet", "csv"], "Sample format")

    config_text = dbutils.widgets.get("config")
    input_mode = dbutils.widgets.get("input_mode")
    keep_latest_per_vin = dbutils.widgets.get("keep_latest_per_vin") == "Yes"
    export_excel = dbutils.widgets.get("export_excel") == "Yes"
    output_dir = dbutils.widgets.get("output_dir")
    sample_path = dbutils.widgets.get("sample_path")
    input_format = dbutils.widgets.get("input_format")
except NameError:
    config_text = default_config_text
    input_mode = DEFAULT_VODR_INPUT_MODE
    keep_latest_per_vin = True
    export_excel = True
    output_dir = DEFAULT_SUBSET_OUTPUT_DIR
    sample_path = DEFAULT_VODR_SAMPLE_PATH
    input_format = DEFAULT_VODR_SAMPLE_FORMAT

config = parse_config_text(config_text)

log_step(f"Config VODR selezionate: {sorted(config)}")
log_step(f"Fat table VODR risolte: {[get_table_path(value) for value in sorted(config)]}")
log_step("Join Mission Test: disabilitato")
log_step(f"Latest per VIN: {keep_latest_per_vin}")
log_step(f"Input mode: {input_mode}")
log_step(f"Export Excel: {export_excel}")
log_step(f"Output dir: {output_dir}")

## 1. Setup Spark e dipendenze Excel

In [ ]:
try:
    spark
except NameError:
    spark = build_spark("iveco-vodr-subset")

if export_excel:
    excel_engine = ensure_excel_writer_available(auto_install=True)
    log_step(f"Engine Excel disponibile: {excel_engine}")
else:
    excel_engine = None
    log_step("Export Excel disabilitato")

## 2. Regole sheet subset

In [ ]:
SUBSET_GROUP_BY = ["product_model"]

SUBSET_REPORT_SHEETS = [
    {
        "sheet_id": "2a",
        "name": "2a Vehicle Weight Time",
        "group_by": SUBSET_GROUP_BY,
        "columns": VODR_PERCENTAGE_GROUPS["2a"],
        "triggers": [0, 0, 0, 1, 1, 1, 1, 1, 1],
        "use_percentage_columns": True,
    },
    {
        "sheet_id": "5a",
        "name": "5a Battery Voltage Data Analysis",
        "group_by": SUBSET_GROUP_BY,
        "columns": VODR_PERCENTAGE_GROUPS["5a"],
        "triggers": [1, 1, 0, 0, 1, 1, 1, 0, 0, 1],
        "use_percentage_columns": True,
    },
    {
        "sheet_id": "5b",
        "name": "5b Battery SOC Data Analysis",
        "group_by": SUBSET_GROUP_BY,
        "columns": VODR_PERCENTAGE_GROUPS["5b"],
        "triggers": [1, 1, 1, 0, 0, 1, 1, 1, 0, 0],
        "use_percentage_columns": True,
    },
    {
        "sheet_id": "5c",
        "name": "5c Battery SOH Data Analysis",
        "group_by": SUBSET_GROUP_BY,
        "columns": VODR_PERCENTAGE_GROUPS["5c"],
        "triggers": [1, 1, 1, 0, 0, 1, 1, 1, 0, 0],
        "use_percentage_columns": True,
    },
    {
        "sheet_id": "6a",
        "name": "6a Cranking Time Data Analysis",
        "group_by": SUBSET_GROUP_BY,
        "columns": VODR_PERCENTAGE_GROUPS["6a"],
        "triggers": [0, 1, 1],
        "use_percentage_columns": True,
    },
    {
        "sheet_id": "average_kick_down",
        "name": "Average Kick down",
        "group_by": SUBSET_GROUP_BY,
        "columns": ["AvgKickDown"],
        "triggers": [0],
        "use_percentage_columns": False,
    },
    {
        "sheet_id": "aebs_intervention",
        "name": "Total number of AEBS Intervention events",
        "group_by": SUBSET_GROUP_BY,
        "columns": ["TotNumAEBSevents"],
        "triggers": [1],
        "use_percentage_columns": False,
    },
    {
        "sheet_id": "level",
        "name": "Low fluid levels",
        "group_by": SUBSET_GROUP_BY,
        "columns": ["OilLevLow", "CoolantLevLow"],
        "triggers": [1, 1],
        "use_percentage_columns": False,
    },
]

log_step(f"Sheet subset configurati: {[sheet['sheet_id'] for sheet in SUBSET_REPORT_SHEETS]}")

## 3. Lettura e preparazione VODR senza Mission Test

In [ ]:
df_vodr_raw = read_vodr_input_data(
    spark=spark,
    input_mode=input_mode,
    sample_path=sample_path,
    input_format=input_format,
    config=config,
)
report_dim(df_vodr_raw, "VODR raw")
report_vin(df_vodr_raw)

df_vodr = clean_spark_column_names(df_vodr_raw)
if "vin" in df_vodr.columns:
    df_vodr = df_vodr.filter(~F.col("vin").isin(VODR_EXCLUDED_VINS))

if keep_latest_per_vin:
    df_vodr = keep_latest_record_per_vin(df_vodr)

df_vodr = add_legacy_preparation_features(df_vodr)
if "TotEngineHours" in df_vodr.columns and "engineminutes" not in df_vodr.columns:
    df_vodr = df_vodr.withColumn("engineminutes", F.col("TotEngineHours").cast("double") * F.lit(60))

if "product_model" not in df_vodr.columns:
    product_like_cols = [col for col in df_vodr.columns if "product" in col.lower() or "model" in col.lower()]
    raise ValueError(f"Colonna product_model assente. Colonne candidate: {product_like_cols}")

report_dim(df_vodr, "VODR prepared subset")
report_vin(df_vodr)
df_vodr.groupBy("product_model").count().orderBy("product_model").show()

## 4. Percentuali richieste

In [ ]:
def add_subset_percentages(df, sheets):
    percentage_sheet_ids = [sheet["sheet_id"] for sheet in sheets if sheet.get("use_percentage_columns")]
    for sheet_id in percentage_sheet_ids:
        columns = VODR_PERCENTAGE_GROUPS[sheet_id]
        present = [col_name for col_name in columns if col_name in df.columns]
        missing = [col_name for col_name in columns if col_name not in df.columns]

        if sheet_id == "2a" and present:
            df = df.fillna(0, subset=present)

        if present:
            df = pyspark_variabili_x(df, present, sheet_id)
            log_step(f"Percentuali subset {sheet_id}: {len(present)} colonne")
        else:
            log_step(f"Percentuali subset {sheet_id}: nessuna colonna presente")

        for col_name in missing:
            log_step(f" - subset {sheet_id} mancante: {col_name}")
    return df


df_subset = add_subset_percentages(df_vodr, SUBSET_REPORT_SHEETS)
report_dim(df_subset, "VODR subset with percentages")

## 5. Build sheet subset

In [ ]:
variables_names = load_variables_names(spark, config)

subset_sheet_outputs = []
skipped_sheet_ids = []

for sheet in SUBSET_REPORT_SHEETS:
    output = build_vodr_sheet_output(df_subset, sheet, variables_names, config)
    if output is None:
        skipped_sheet_ids.append(sheet["sheet_id"])
        continue
    subset_sheet_outputs.append(output)

log_step(f"Sheet subset generati: {[output['sheet_id'] for output in subset_sheet_outputs]}")
if skipped_sheet_ids:
    log_step(f"Sheet subset saltati: {skipped_sheet_ids}")

for output in subset_sheet_outputs:
    log_step(f"Preview subset {output['sheet_id']} - {output['sheet_name']}")
    try:
        display(output["dataframe"].head(20))
    except NameError:
        print(output["dataframe"].head(20))

## 6. Export Excel

In [ ]:
subset_excel_path = None
if export_excel:
    config_str = "_".join(map(str, sorted(int(value) for value in config)))
    excel_file_name = f"VODR_subset_{config_str}.xlsx"
    subset_excel_path = export_excel_outputs(subset_sheet_outputs, output_dir, excel_file_name)
    log_step(f"Excel subset generato: {subset_excel_path}")
else:
    log_step("Export Excel subset saltato")

try:
    dbutils
    if subset_excel_path:
        copy_result = copy_excel_to_dbfs(
            subset_excel_path,
            dbutils,
            spark=spark,
            dbfs_output_dir=DEFAULT_SUBSET_DBFS_OUTPUT_DIR,
        )
        print(copy_result)
except NameError:
    log_step("dbutils non disponibile: copia DBFS saltata")

In [ ]:
log_step("Vodr_subset completato")